# NLP with Pre-trained Language Models

**MIDAS AI in Research Handbook — Chapter 17b**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/bert_nlp_demo.ipynb)

This notebook covers three common NLP tasks using pre-trained language models from the Hugging Face ecosystem:

1. **Named Entity Recognition (NER)** — identifying people, organizations, and locations in text
2. **Semantic Similarity** — measuring how alike two passages are in meaning
3. **Text Classification** — categorizing text with a fine-tuned model

The companion chapter explains the concepts behind each task. This notebook is where you run the code.

**Before you start:** Go to **Runtime → Change runtime type → T4 GPU**, then run all cells.


## Step 1 — Install Libraries

This takes about two to three minutes on a fresh Colab session. Run it once.

In [ ]:
# Install transformers and sentence-transformers
# The -q flag suppresses most of the output
!pip install -q transformers==4.40.2 sentence-transformers==3.0.1 datasets accelerate


## Step 2 — Check for GPU

The code below detects whether a GPU is available and sets the `device` variable accordingly. Both libraries use this automatically.

In [ ]:
import torch

if torch.cuda.is_available():
    device = 0  # first GPU
    device_name = torch.cuda.get_device_name(0)
    print(f"GPU detected: {device_name}")
else:
    device = -1  # CPU fallback
    print("No GPU found — running on CPU. Some cells will be slower.")


---

## Part 1 — Named Entity Recognition (NER)

Named entity recognition finds spans of text that refer to named things and labels them by type. The model we use here is `dslim/bert-base-NER`, which is BERT fine-tuned on the CoNLL-2003 benchmark. It recognizes four entity types:

- **PER** — person names
- **ORG** — organizations, companies, agencies
- **LOC** — locations: countries, cities, geographic features
- **MISC** — miscellaneous: nationalities, event names, product names


### Load the NER Pipeline

In [ ]:
from transformers import pipeline

ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=device
)
print("NER pipeline ready.")


### Run NER on Sample Texts

The texts below are short passages from different research domains. We will look at what the model picks up and whether any results look off.


In [ ]:
texts = [
    "The EPA released a report last week from its regional office in Chicago, citing Midwest Energy as a repeat violator.",
    "Dr. Priya Mehta, a senior fellow at the Brookings Institution, presented findings to the Senate Committee on Environment.",
    "The World Health Organization and the African Union signed a partnership agreement in Addis Ababa on Thursday.",
    "Stanford University and MIT researchers published a joint paper on carbon capture in the journal Nature Climate Change.",
    "The Federal Reserve Bank of New York noted that inflation in the eurozone remains elevated despite ECB intervention.",
]

for text in texts:
    print(f"TEXT: {text}")
    entities = ner(text)
    for ent in entities:
        print(f"  [{ent['entity_group']}]  '{ent['word']}'  (score: {ent['score']:.3f})")
    print()


### Aggregate Entities Across a Corpus

In practice, you want counts across many documents. The cell below applies NER to all texts and builds a frequency table for each entity type.


In [ ]:
from collections import defaultdict, Counter

entity_store = defaultdict(list)

for text in texts:
    for ent in ner(text):
        entity_store[ent["entity_group"]].append(ent["word"])

print("Top entities by type:")
for etype in ["ORG", "PER", "LOC", "MISC"]:
    counts = Counter(entity_store[etype])
    top = counts.most_common(5)
    print(f"\n  {etype}:")
    for name, count in top:
        print(f"    {name}: {count}")


### Exercise 1

Replace the `texts` list above with three to five short passages from your own research domain. Then answer these questions before moving on:

1. Which entity types does the model handle well in your domain?
2. Are there any systematic errors (for example, does it misclassify a technical term as a person name)?
3. Search the [Hugging Face Model Hub](https://huggingface.co/models?pipeline_tag=token-classification&search=NER) for a model fine-tuned on text closer to your domain. Common search terms: "NER biomedical", "NER legal", "NER scientific". Try running it on the same passages using the same pipeline call, just changing the `model=` argument.


---

## Part 2 — Semantic Similarity

Semantic similarity measures how alike two pieces of text are in meaning, independent of shared wording. Two sentences can be phrased completely differently and still express the same idea.

We use `sentence-transformers` with the model `all-MiniLM-L6-v2`. This model is optimized for producing sentence embeddings that are directly useful for comparison. The measure we compute is cosine similarity, which ranges from 0 (unrelated) to 1 (nearly identical in meaning).


### Load the Sentence Transformer

In [ ]:
from sentence_transformers import SentenceTransformer, util

sent_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if device == 0 else "cpu")
print("Sentence transformer ready.")


### Compute Pairwise Similarity

In [ ]:
passages = [
    "Carbon emissions near the industrial district increased substantially last year.",
    "Air pollution levels around the manufacturing zone rose sharply over the past twelve months.",
    "The company posted its strongest quarterly revenue figures in five years.",
    "Greenhouse gas output from factories in the region climbed significantly in recent months.",
    "The quarterly earnings report exceeded analyst expectations by a wide margin.",
]

embeddings = sent_model.encode(passages, convert_to_tensor=True)
sim_matrix = util.cos_sim(embeddings, embeddings).cpu().numpy()

print("Cosine similarity matrix (rows = passages, columns = passages):")
print()
header = "       " + "  ".join([f"P{i+1}" for i in range(len(passages))])
print(header)
for i, row in enumerate(sim_matrix):
    scores = "  ".join([f"{s:.2f}" for s in row])
    print(f"  P{i+1}   {scores}")

print()
print("Passages:")
for i, p in enumerate(passages):
    print(f"  P{i+1}: {p}")


### Visualize as a Heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 6))

labels = [f"P{i+1}" for i in range(len(passages))]
im = ax.imshow(sim_matrix, vmin=0, vmax=1, cmap="YlOrRd")
plt.colorbar(im, ax=ax, label="Cosine similarity")

ax.set_xticks(range(len(passages)))
ax.set_yticks(range(len(passages)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

for i in range(len(passages)):
    for j in range(len(passages)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                fontsize=10, color="black" if sim_matrix[i, j] < 0.7 else "white")

ax.set_title("Pairwise Semantic Similarity", fontsize=13)
plt.tight_layout()
plt.savefig("similarity_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()
print("Heatmap saved to similarity_heatmap.png")


### Find the Most and Least Similar Pairs


In [ ]:
import numpy as np

n = len(passages)
pairs = []
for i in range(n):
    for j in range(i + 1, n):
        pairs.append((sim_matrix[i, j], i, j))

pairs.sort(reverse=True)

print("Top 3 most similar pairs:")
for score, i, j in pairs[:3]:
    print(f"  [{score:.3f}]  P{i+1} vs P{j+1}")
    print(f"    P{i+1}: {passages[i]}")
    print(f"    P{j+1}: {passages[j]}")
    print()

print("Top 3 least similar pairs:")
for score, i, j in pairs[-3:]:
    print(f"  [{score:.3f}]  P{i+1} vs P{j+1}")
    print(f"    P{i+1}: {passages[i]}")
    print(f"    P{j+1}: {passages[j]}")
    print()


### Exercise 2

Substitute the `passages` list with text from your own work: open-ended survey responses, abstract excerpts, interview segments, or anything where you want to understand thematic grouping.

Then try this:

1. Find the highest-scoring pair. Does that result make sense? Are those two passages genuinely similar in meaning, or is the model responding to shared vocabulary?
2. Pick two passages that you know express opposite positions on the same topic (for example, one supportive of a policy and one critical of it). What similarity score do they get? Explain why the score is what it is.
3. What threshold would you use to call two passages "similar enough to treat as duplicates"? There is no single right answer — it depends on how strict you need to be. Try a few and see what the implied groupings look like.


---

## Part 3 — Text Classification

Text classification assigns a label to each piece of text from a fixed set of categories. The model here is `distilbert-base-uncased-finetuned-sst-2-english`, a compact BERT variant fine-tuned on the Stanford Sentiment Treebank for binary sentiment classification (POSITIVE / NEGATIVE).

This section also demonstrates how to compare a fine-tuned model against a zero-shot classifier on the same texts, so you can see the trade-offs directly.


### Load the Fine-Tuned Classifier

In [ ]:
sentiment = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)
print("Sentiment classifier ready.")


In [ ]:
sample_texts = [
    "The data collection process was frustrating and poorly documented.",
    "The training session was clear and well organized.",
    "Results were mixed, with some improvements but also notable regressions.",
    "Participants reported that the intervention had no meaningful effect on their behavior.",
    "The field site conditions were ideal, and data quality exceeded our expectations.",
    "We were unable to replicate the original findings despite following the protocol exactly.",
]

print("Fine-tuned sentiment classifier results:")
print()
results = sentiment(sample_texts)
for text, result in zip(sample_texts, results):
    label = result["label"]
    score = result["score"]
    print(f"  [{label} {score:.3f}]  {text}")


### Compare with Zero-Shot Classification

Zero-shot classification lets you define your own labels without any retraining. The model we use here is `facebook/bart-large-mnli`, which appeared in Chapter 17. We will run the same texts through both models and compare where they agree and disagree.


In [ ]:
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

# Use labels that map loosely to positive/negative for comparison
candidate_labels = ["positive outcome", "negative outcome", "neutral or ambiguous"]

print("Zero-shot classification results:")
print()
for text in sample_texts:
    zs_result = zero_shot(text, candidate_labels=candidate_labels)
    top_label = zs_result["labels"][0]
    top_score = zs_result["scores"][0]
    print(f"  [{top_label} {top_score:.3f}]  {text}")


In [ ]:
# Side-by-side comparison
print(f"{'Text':<60}  {'Fine-tuned':>12}  {'Zero-shot':>20}")
print("-" * 96)

ft_results = sentiment(sample_texts)
zs_results = [zero_shot(t, candidate_labels=candidate_labels) for t in sample_texts]

for text, ft, zs in zip(sample_texts, ft_results, zs_results):
    ft_str = f"{ft['label']} ({ft['score']:.2f})"
    zs_str = f"{zs['labels'][0].split()[0].upper()} ({zs['scores'][0]:.2f})"
    display_text = text[:57] + "..." if len(text) > 60 else text
    print(f"{display_text:<60}  {ft_str:>12}  {zs_str:>20}")


### Exercise 3

1. Look at the comparison table. Find at least one text where the two models disagree. Why might they disagree? Which one do you think is more correct for that case?

2. The fine-tuned model was trained on movie review sentiment. Try adding a text from your own research domain that has clear positive or negative valence but uses domain-specific vocabulary. Does the model handle it correctly?

3. Try changing the `candidate_labels` for the zero-shot classifier to labels that match your own research categories (for example, `["supports the policy", "opposes the policy", "neutral"]`). Run it on a few of your own texts. Does the label phrasing affect the results? Try rewording a label and compare.


---

## Bonus — Applying NER to a Larger Document

The cell below applies NER to a longer passage paragraph by paragraph, then aggregates entity counts. This is closer to what a real research workflow would look like.


In [ ]:
document = """
The Environmental Protection Agency announced new air quality standards last Tuesday,
drawing immediate responses from the American Petroleum Institute and the National
Resources Defense Council. EPA Administrator Michael Regan stated that the rules
represent the most significant update since the Obama administration.

The new standards apply to particulate matter and ozone levels in metropolitan areas
including Los Angeles, Houston, and Chicago. Critics from the Texas Oil and Gas
Association argue that the regulations could cost the industry billions of dollars
in compliance costs over the next decade.

Supporters, including Senator Ed Markey of Massachusetts, called the rules long overdue.
The World Resources Institute and the Sierra Club both issued statements welcoming
the announcement. The rules are expected to face legal challenges from industry
groups in the Fifth Circuit Court of Appeals.
"""

paragraphs = [p.strip() for p in document.strip().split("\n\n") if p.strip()]

all_entities = defaultdict(list)
for para in paragraphs:
    for ent in ner(para):
        all_entities[ent["entity_group"]].append(ent["word"])

print("Entity counts across document:")
for etype in ["ORG", "PER", "LOC", "MISC"]:
    if all_entities[etype]:
        counts = Counter(all_entities[etype])
        print(f"\n  {etype}:")
        for name, count in counts.most_common():
            print(f"    {name}: {count}")


---

## Summary

| Task | Model | Library | Use when |
|---|---|---|---|
| Named entity recognition | `dslim/bert-base-NER` | `transformers` | You need to find who/what appears in text |
| Semantic similarity | `all-MiniLM-L6-v2` | `sentence-transformers` | You want to compare meaning across documents |
| Sentiment / classification | `distilbert-base-uncased-finetuned-sst-2-english` | `transformers` | A pre-existing model matches your task closely |
| Zero-shot classification | `facebook/bart-large-mnli` | `transformers` | Your categories are custom and you have no labeled data |

**Next steps:**

- Browse [huggingface.co/models](https://huggingface.co/models) for models fine-tuned on text from your domain
- For corpora too large for Colab, look into batch processing with `pipeline(batch_size=...)` or moving to Great Lakes with a GPU node
- If a zero-shot classifier performs inconsistently, collecting 50–100 labeled examples enables fine-tuning on your specific task

**Questions or feedback?** [Open an issue on GitHub](https://github.com/xiaosuhu/midas-ai-in-research/issues)
